In [1]:
# 1 Importing libraries for functions
import requests
import pandas as pd
from sklearn.preprocessing import StandardScaler
# Note: LinearRegression and r2_score are no longer strictly needed for percentile,
# but keeping them imported as they were in your original script.
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

# 2 All Lists used in the code
team_ids = [
    108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121,
    133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 158
]
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}

# Master DataFrames
all_standard = []
all_statcast = []

# Statcast features and betas
hit_features = ['Whiff %','Batted Balls','Hard Hit %','Exit Velocity','XBA','Zone Contact %','Chase %', 'Zone Swing %', 'Straight %']
walk_features = ['Batted Balls']
bats_features = ['Pitches', 'Batted Balls']
bases_features = ['XSLG','Whiff %','Chase %','Barrel %', 'Hard Hit %', 'Exit Velocity', 'Launch Angle', 'Straight %', 'Batted Balls']

hit_betas = {'Whiff %': .6,'Batted Balls': .8,'Hard Hit %': 1,'Exit Velocity': .7,'XBA': .9,'Zone Contact %': .3,'Chase %': .5, 'Zone Swing %': .2, 'Straight %': .4}
walk_betas = {'Batted Balls': 1}
bats_betas = {'Pitches': .3, 'Batted Balls': .7}
bases_betas = {'XSLG': .9,'Whiff %': .5,'Chase %': .5,'Barrel %': 1, 'Hard Hit %': .8, 'Exit Velocity': .7, 'Launch Angle': .3, 'Straight %': .4, 'Batted Balls': .3}

unwanted = ['Unnamed: 1_level_0', 'Unnamed: 0_level_0', 'Statcast', 'Standard Stats','Unnamed: 26_level_1']

# List of team nicknames for filtering players later
team_nicknames = [
    'D-Backs', 'Braves', 'Orioles', 'Red Sox', 'Cubs', 'White Sox', 'Reds', 'Guardians',
    'Rockies', 'Tigers', 'Astros', 'Royals', 'Angels', 'Dodgers', 'Marlins', 'Brewers',
    'Twins', 'Yankees', 'Athletics', 'Phillies', 'Pirates', 'Padres', 'Giants', 'Mariners',
    'Cardinals', 'Rays', 'Rangers', 'Blue Jays', 'Nationals', 'Mets'
]
# Regex pattern to identify non-player rows (like team totals)
pattern = r'MLB|' + '|'.join(team_nicknames)

# 3 Helper functions
def clean_column(col):
    for u in unwanted:
        col = col.replace(u, '').strip()
    col = ' '.join(col.split())
    return col

def compute_statcast_score(df, features, betas):
    """Computes a weighted sum of scaled features."""
    # Ensure all features exist and handle potential NaNs by filling with 0 before scaling
    # (or you might choose a different strategy for NaNs depending on your data)
    for feat in features:
        if feat not in df.columns:
            df[feat] = 0 # Add missing feature with 0
    
    # Scale only the relevant features
    scaler = StandardScaler()
    scaled_features_df = pd.DataFrame(scaler.fit_transform(df[features].fillna(0)), columns=features, index=df.index)

    score = 0
    for feat in features:
        score += scaled_features_df[feat] * betas[feat]
    return score

def flip_name(name):
    """Converts "Last, First" to "First Last" and removes commas."""
    if ',' in name:
        last, first = name.split(',', 1)
        return first.strip() + ' ' + last.strip()
    return name.strip()

# 4 Fetching the data for the current season (2024)
current_season = 2024 # Keeping it as 2024 as per your paste.txt

for team_id in team_ids:
    url = f"https://baseballsavant.mlb.com/team/{team_id}?view=statcast&nav=hitting&season={current_season}"
    response = requests.get(url, headers=headers)
    try:
        tables = pd.read_html(response.text)
        if len(tables) < 3:
            print(f"Team {team_id}: Not enough tables found, skipping.")
            continue
        standard = tables[0]
        statcast1 = tables[1]
        statcast2 = tables[2]
    except Exception as e:
        print(f"Team {team_id}: Error reading tables: {e}")
        continue

    # Clean column names for standard stats
    standard.columns = [' '.join(col).strip() if isinstance(col, tuple) else col for col in standard.columns]
    # Clean column names for statcast tables
    statcast1.columns = [str(col) for col in statcast1.columns]
    statcast2.columns = [str(col) for col in statcast2.columns]

    for df in [standard, statcast1, statcast2]:
        new_columns = {}
        for col in df.columns:
            col_str = ' '.join(col).strip() if isinstance(col, tuple) else str(col)
            new_columns[col] = clean_column(col_str)
        df.rename(columns=new_columns, inplace=True)

    # Drop specific columns that cause issues or are not needed
    # Ensure 'Season' column is not dropped if it's implicitly part of the column names (it usually isn't)
    drop_cols_statcast1 = [col for col in ['Season','Pitches'] if col in statcast1.columns]
    drop_cols_statcast2 = [col for col in ['Season','Barrel %'] if col in statcast2.columns]
    # These drops seem to be intended to remove specific columns from 'standard' to create 'statcast3'
    # and then to simplify 'standard' itself.
    drop_cols_standard_for_statcast3 = [col for col in ['Season','AB','H','2B','3B','HR','BB','PA','SO','BA','OBP','SLG','WOBA','WOBACON'] if col in standard.columns]
    drop_cols_standard_final = [col for col in ['Season','PA','SO','BA','OBP','SLG','WOBA','WOBACON','Pitches','Batted Balls','Barrels','Barrel %','Hard Hit %','Exit Velocity','Launch Angle','XBA','XSLG','XWOBA','XWOBACON'] if col in standard.columns]
    
    statcast1 = statcast1.drop(columns=drop_cols_statcast1, errors='ignore')
    statcast2 = statcast2.drop(columns=drop_cols_statcast2, errors='ignore')
    statcast3 = standard.drop(columns=drop_cols_standard_for_statcast3, errors='ignore')
    standard = standard.drop(columns=drop_cols_standard_final, errors='ignore') # Keep only 'Player', '2B', '3B', 'HR', 'H', 'BB', 'AB'

    # Rename player column if it's not simply 'Player' (e.g., 'Player Name')
    player_col_statcast3 = [col for col in statcast3.columns if 'Player' in col][0]
    statcast3 = statcast3.rename(columns={player_col_statcast3: 'Player'})

    # Merge statcast tables
    statcast = pd.merge(statcast1, statcast2, on='Player', how='inner')
    statcast = pd.merge(statcast, statcast3, on='Player', how='inner')

    # Calculate Total Bases (TB) and Runs for standard stats
    for col in ['2B', '3B', 'HR', 'H', 'AB', 'BB']:
        if col not in standard.columns:
            standard[col] = 0 # Ensure columns exist before calculation
    standard['TB'] = (standard['2B'] * 2 + standard['3B'] * 3 + standard['HR'] * 4 +
                      (standard['H'] - standard['2B'] - standard['3B'] - standard['HR']))
    # Avoid division by zero for Runs calculation
    standard['Runs'] = standard.apply(lambda row: ((row['H'] + row['BB']) * row['TB']) / (row['AB'] + row['BB']) if (row['AB'] + row['BB']) != 0 else 0, axis=1)
    standard['Runs'] = standard['Runs'].round()

    # Add TeamID
    standard['TeamID'] = team_id
    statcast['TeamID'] = team_id

    # Compute Statcast scores (Hit_Score, Walk_Score, etc.)
    # Ensure all required features are in statcast or handled by compute_statcast_score
    statcast['Hit_Score']   = compute_statcast_score(statcast.copy(), hit_features, hit_betas) # Pass copy to avoid modifying original df in function
    statcast['Walk_Score']  = compute_statcast_score(statcast.copy(), walk_features, walk_betas)
    statcast['Bats_Score']  = compute_statcast_score(statcast.copy(), bats_features, bats_betas)
    statcast['Bases_Score'] = compute_statcast_score(statcast.copy(), bases_features, bases_betas)
    
    statcast['Hit_Score']   = statcast['Hit_Score'].round(3)
    statcast['Walk_Score']  = statcast['Walk_Score'].round(3)
    statcast['Bats_Score']  = statcast['Bats_Score'].round(3)
    statcast['Bases_Score'] = statcast['Bases_Score'].round(3)

    all_standard.append(standard)
    all_statcast.append(statcast)        

# 5 Concatenate all teams' data
df_standard_all = pd.concat(all_standard, ignore_index=True)
df_statcast_all = pd.concat(all_statcast, ignore_index=True)

# 6 Remove rows where 'Player' contains 'MLB' or any team name/nickname (team totals)
df_standard_all = df_standard_all[~df_standard_all['Player'].str.contains(pattern, case=False, na=False)]
df_statcast_all = df_statcast_all[~df_statcast_all['Player'].str.contains(pattern, case=False, na=False)]

# 7 Fix player names to "First Last" (no commas)
df_standard_all['Player'] = df_standard_all['Player'].apply(flip_name)
df_statcast_all['Player'] = df_statcast_all['Player'].apply(flip_name)

# 8 Export to CSV (Standard.csv and Statcast.csv)
df_standard_all = df_standard_all.drop(columns=['TeamID'], errors='ignore')
df_statcast_all = df_statcast_all.drop(columns=['TeamID'], errors='ignore')
df_standard_all.to_csv('Standard.csv', index=False)
df_statcast_all.to_csv('Statcast.csv', index=False)

# 9 Merge on Player to align scores with real stats
# This is the corrected line where the comma issue was
df_merged = pd.merge(
    df_statcast_all[['Player', 'Hit_Score', 'Walk_Score', 'Bats_Score', 'Bases_Score']],
    df_standard_all[['Player', 'H', 'BB', 'AB', 'TB']],
    on='Player', # Corrected: 'on' should not be a list if only one column is specified, or remove the trailing comma if it was a list [something,]
    how='inner'
)

# Only keep players with more than 100 AB
df_merged = df_merged[df_merged['AB'] > 100]

# 10 Calculate percentile ranks for each Statcast score (1=worst, 100=best)
for score_col in ['Hit_Score', 'Walk_Score', 'Bats_Score', 'Bases_Score']:
    # Ensure no NaNs before ranking, fill with 0 or a sensible value if present
    df_merged[f'{score_col}_pct'] = df_merged[score_col].fillna(0).rank(pct=True) * 100
    df_merged[f'{score_col}_pct'] = df_merged[f'{score_col}_pct'].round(1)

# 11 Calculate actual Runs (already done during data fetching, just ensuring it's in df_merged)
# The Runs calculation logic was in the loop, ensure it's propagated to df_merged
# If 'Runs' is not in df_merged from standard_all, re-calculate based on H, BB, TB, AB from df_merged
if 'Runs' not in df_merged.columns:
    df_merged['Runs'] = df_merged.apply(lambda row: ((row['H'] + row['BB']) * row['TB']) / (row['AB'] + row['BB']) if (row['AB'] + row['BB']) != 0 else 0, axis=1)

# 12 Calculate Expected Runs (xRuns) based on the mean of the percentile scores
df_merged['xRuns'] = df_merged[['Hit_Score_pct', 'Walk_Score_pct', 'Bats_Score_pct', 'Bases_Score_pct']].mean(axis=1)
df_merged['xRuns'] = df_merged['xRuns'].round(1)

# 13 Add expected runs tier (percentile 1-100)
df_merged['xRuns_Tier'] = df_merged['xRuns'].rank(pct=True) * 100
df_merged['xRuns_Tier'] = df_merged['xRuns_Tier'].round(1)

# 14 Clean up and Export final predictions
# Remove the original raw score columns as they are replaced by percentiles
score_cols_to_drop = ['Hit_Score', 'Walk_Score', 'Bats_Score', 'Bases_Score']
df_merged = df_merged.drop(columns=score_cols_to_drop, errors='ignore')

# Round final Runs/xRuns for display
df_merged[['Runs', 'xRuns']] = df_merged[['Runs', 'xRuns']].round(1)

df_merged.to_csv('Predictions_percentile.csv', index=False) # Renamed output file to reflect percentile nature


C:\Users\nncg7\AppData\Local\Temp\ipykernel_21580\3987294140.py:83: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)
C:\Users\nncg7\AppData\Local\Temp\ipykernel_21580\3987294140.py:83: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)
C:\Users\nncg7\AppData\Local\Temp\ipykernel_21580\3987294140.py:83: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)
C:\Users\nncg7\AppData\Local\Temp\ipykernel_21580\3987294140.py:83: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version

Team 117: Error reading tables: No tables found


C:\Users\nncg7\AppData\Local\Temp\ipykernel_21580\3987294140.py:83: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)
C:\Users\nncg7\AppData\Local\Temp\ipykernel_21580\3987294140.py:83: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)
C:\Users\nncg7\AppData\Local\Temp\ipykernel_21580\3987294140.py:83: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)
C:\Users\nncg7\AppData\Local\Temp\ipykernel_21580\3987294140.py:83: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version